# PhysAR-IDS -- NB17 (v6): Run-held-out evaluation on the TU Delft digital-substation dataset

**Purpose.** The Sweeten data has one session per class, so class and session are confounded. This dataset (Cyber-Physical Power System Dataset for Cyber Security of Digital Substations, TU Delft, Zenodo 15371179, CC BY 4.0) has **repeated runs** and, in the GOOSE-injection runs, a **normal period and an attack period inside the same run**. This notebook builds windowed features and tests detection on **held-out runs**.

**Data used (v1).** `Baseline` (5 runs, normal), `Fault` (2 variants x 5 runs, electrical disturbance, no attack), `GOOSE Injection` (6 variants x 5 runs). Not used yet: `Reconnaissance` and `SV Injection` (one run per variant, nothing to hold out) and `Resource Exhaustion` (no frame labels).

**Windows and labels (0.5 s windows).**
* `attack` (2): window overlaps the period from the first to the last injected GOOSE frame (from the per-frame `*_labels.csv` and the pcap timestamps).
* `fault` (1): the RTDS fault-input signal is on and the window is not an attack window. `FLT_input` is used for labelling only and is **never** a feature.
* `normal` (0): everything else.
* Clock alignment: the `rscad` clock is Dutch local time and the pcap is UTC. The offset is estimated per run (expected -2 h) and every run's residual is printed; runs with a residual above 5 s or a frame/label mismatch are listed.

**Features.** Electrical (per-window RMS, mean, std and peak-to-peak of the 3-phase current and voltage, breaker state and trip signals); host (CPU, RAM, network rates from the 2 s resource log); packets (frame, byte, GOOSE, SV and other counts, distinct sources and destinations, inter-arrival statistics).

**Splits.** `heldout_runs`: runs 1-3 train, runs 4-5 test, for every variant. `random_windows`: stratified 70/30 over windows (leakage control; neighbouring windows are near-duplicates, so expect it to inflate scores). Seeds [42, 7, 123, 2024, 99]; XGBoost and random forest; feature-group ablation with paired t-tests and Holm correction; 95% interval from a bootstrap over **runs**.

**Caveats to state if you use it.** Attack windows are few (about 64 injected frames per run), so per-class F1 is noisy. The attack label covers the whole injection period, including gaps between injected frames. The packet features are header-level counts, not decoded GOOSE fields.

Set `ROOT` to the `Scenarios` folder. Needs `dpkt` (`pip install dpkt`). Outputs are written to `nb17_out` next to the dataset; check free disk space first (feature tables are small).

In [ ]:
import os, time, json, warnings, glob, re, struct
import numpy as np, pandas as pd, joblib
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import train_test_split
import xgboost as xgb
warnings.filterwarnings('ignore')
PIPELINE_VERSION = 'v6'
SEEDS = [int(s) for s in os.environ.get('NB17_SEEDS', '42,7,123,2024,99').split(',')]
ROOT = os.environ.get('NB17_ROOT', r'C:\Users\user\Downloads\2026_03_17_CPPS_Dataset_CyberSec_DigitalSubstation_TUD\2026_03_17_CPPS_Dataset_CyberSec_DigitalSubstation_TUD\Scenarios')
OUT = os.environ.get('NB17_OUT', os.path.join(os.path.dirname(ROOT), 'nb17_out')); os.makedirs(OUT, exist_ok=True)
WIN = 0.5                                   # seconds per window
INCLUDE = ['Baseline', 'Fault', 'GOOSE Injection']   # runs with labels or no attack. Reconnaissance / SV Injection have 1 run per variant; Resource Exhaustion has no frame labels.
TRAIN_RUNS, TEST_RUNS = {1, 2, 3}, {4, 5}
XGB_P = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.6, min_child_weight=3, reg_lambda=1.0, tree_method='hist', n_jobs=-1, verbosity=0)
print('root', ROOT, '| out', OUT, '| seeds', SEEDS)


In [ ]:
try:
    import dpkt
except ImportError:
    import subprocess, sys; subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'dpkt']); import dpkt
def longpath(p):                                  # Windows MAX_PATH (260) workaround for the long GOOSE-injection folder names
    p = os.path.abspath(p)
    return ('\\\\?\\' + p) if os.name == 'nt' and not p.startswith('\\\\?\\') else p
ELEC_SIG = ['I1_a', 'I1_b', 'I1_c', 'V1_a', 'V1_b', 'V1_c']; ELEC_DISC = ['CB1_state', 'ied_trip_signal', 'vied_trip_signal']
HOST_NUM = ['CPU 1 Usage (%)', 'CPU 2 Usage (%)', 'CPU 3 Usage (%)', 'CPU 4 Usage (%)', 'RAM Usage (%)', '%CPU', '%MEM']
def find_runs(root):
    runs = []
    for dp, dn, fn in os.walk(root):
        if len([f for f in fn if f.endswith('_rscad.csv')]) == 1: runs.append(dp)
    return sorted(runs)
def run_meta(path, root):
    rel = os.path.relpath(path, root).split(os.sep); m = re.search(r'(\d+)$', rel[-1])
    return dict(scenario=rel[0], variant=rel[1] if len(rel) >= 3 else rel[0], run=int(m.group(1)) if m else -1)
def read_pcap(path):
    import dpkt
    ts, ln, et, src, dst = [], [], [], [], []
    with open(path, 'rb') as fh:
        try: rd = dpkt.pcap.Reader(fh)
        except Exception: fh.seek(0); rd = dpkt.pcapng.Reader(fh)
        for t, b in rd:
            ts.append(t); ln.append(len(b))
            if len(b) >= 14:
                e = (b[12] << 8) | b[13]
                if e == 0x8100 and len(b) >= 18: e = (b[16] << 8) | b[17]
                et.append(e); dst.append(int.from_bytes(b[0:6], 'big')); src.append(int.from_bytes(b[6:12], 'big'))
            else: et.append(0); dst.append(0); src.append(0)
    return np.array(ts, float), np.array(ln), np.array(et), np.array(src), np.array(dst)
def pick(folder, pat, required=True, avoid=None):
    f = sorted(glob.glob(os.path.join(folder, pat)))
    if avoid: f = [x for x in f if avoid not in os.path.basename(x)] or f
    if not f:
        if required: raise FileNotFoundError((folder, pat))
        return None
    return longpath(f[0])
def build_run(folder, win=WIN):
    rs = pd.read_csv(pick(folder, '*_rscad.csv'), low_memory=False); t0 = pd.Timestamp(rs['formatted_datetime'].iloc[0])
    t = (rs['time'] - rs['time'].iloc[0]).values; widx = np.floor(t / win).astype(int); nW = int(widx.max()) + 1
    out = pd.DataFrame(index=pd.RangeIndex(nW)); out.index.name = 'win'; g = rs.groupby(widx)
    for s in ELEC_SIG:
        out[f'e_{s}_rms'] = np.sqrt((rs[s] ** 2).groupby(widx).mean()); out[f'e_{s}_mean'] = g[s].mean(); out[f'e_{s}_std'] = g[s].std(); out[f'e_{s}_ptp'] = g[s].max() - g[s].min()
    for s in ELEC_DISC: out[f'e_{s}_max'] = g[s].max(); out[f'e_{s}_min'] = g[s].min()
    fault = (g['FLT_input'].max() > 0.5).values                                      # used for labels only, never as a feature
    hp = pick(folder, 'resource_monitor*.csv', required=False)
    hcols = [f'h_{c}' for c in HOST_NUM + ['net_tx_rate', 'net_rx_rate']]; host = pd.DataFrame(index=out.index, columns=hcols, dtype=float)
    if hp:
        h = pd.read_csv(hp)
        for c in ['%CPU', '%MEM']: h[c] = h[c].astype(str).str.replace(',', '.').astype(float)
        h['_t'] = (pd.to_datetime(h['Timestamp']) - t0).dt.total_seconds(); dt = h['_t'].diff()
        h['net_tx_rate'] = h['Network Sent (GiB)'].diff() / dt; h['net_rx_rate'] = h['Network Received (GiB)'].diff() / dt
        m = pd.merge_asof(pd.DataFrame({'_t': np.arange(nW) * win}), h.sort_values('_t'), on='_t', direction='backward')
        for c in HOST_NUM + ['net_tx_rate', 'net_rx_rate']: host[f'h_{c}'] = m[c].values
    out = out.join(host)
    pp = pick(folder, '*.pcap', avoid='attacker'); lp = pick(folder, '*_labels.csv', required=False, avoid='attacker')
    ts, ln, et, src, dst = read_pcap(pp)
    r0 = (t0 - pd.Timestamp('1970-01-01')).total_seconds(); diff = ts[0] - r0; tz_h = round(diff / 3600); resid = diff - tz_h * 3600
    tp = ts - (r0 + tz_h * 3600); keep = (tp >= 0) & (tp < nW * win)
    df = pd.DataFrame(dict(w=np.floor(tp / win).astype(int), len=ln, et=et, src=src, dst=dst, ts=tp))[keep]; gp = df.groupby('w')
    out['p_n'] = gp.size(); out['p_bytes'] = gp['len'].sum(); out['p_len_mean'] = gp['len'].mean(); out['p_len_max'] = gp['len'].max()
    out['p_n_goose'] = df[df.et == 0x88B8].groupby('w').size(); out['p_n_sv'] = df[df.et == 0x88BA].groupby('w').size()
    out['p_n_other'] = df[~df.et.isin([0x88B8, 0x88BA])].groupby('w').size(); out['p_n_src'] = gp['src'].nunique(); out['p_n_dst'] = gp['dst'].nunique()
    df = df.sort_values('ts'); df['_ia'] = df['ts'].diff(); gi = df.groupby('w')['_ia']
    out['p_ia_mean'] = gi.mean(); out['p_ia_std'] = gi.std(); out['p_ia_max'] = gi.max()
    for c in ['p_n', 'p_bytes', 'p_n_goose', 'p_n_sv', 'p_n_other']: out[c] = out[c].fillna(0)
    out = out.fillna(0); attack = np.zeros(nW, bool); n_att = 0; lab_ok = lp is None
    if lp is not None:
        lab = pd.read_csv(lp)
        if len(lab) == len(ts):
            a = lab['label'].values != 0; n_att = int(a.sum()); lab_ok = True
            if a.any(): ws = np.arange(nW) * win; attack = (ws + win > tp[a].min()) & (ws < tp[a].max())    # window overlaps the first-to-last injected frame
        else: n_att = -1
    out['y'] = np.where(attack, 2, np.where(fault, 1, 0))
    info = dict(n_win=nW, n_frames=len(ts), tz_hours=tz_h, tz_resid_s=round(float(resid), 3), has_labels=lp is not None, labels_match=lab_ok, attack_frames=n_att, attack_win=int(attack.sum()), fault_win=int((fault & ~attack).sum()))
    return out.reset_index(), info


In [ ]:
runs = [r for r in find_runs(ROOT) if run_meta(r, ROOT)['scenario'] in INCLUDE]; print(len(runs), 'runs under', INCLUDE)
tabs, infos = [], []; t0 = time.time()
for i, r in enumerate(runs):
    m = run_meta(r, ROOT); cache = os.path.join(OUT, f"feat_{m['scenario']}_{m['variant']}_{m['run']}.csv".replace(' ', '_'))
    try:
        if os.path.exists(cache): f = pd.read_csv(cache); info = json.load(open(cache + '.json'))
        else:
            f, info = build_run(r); f.to_csv(cache, index=False); json.dump({k: (int(v) if isinstance(v, (np.integer,)) else v) for k, v in info.items()}, open(cache + '.json', 'w'))
        for k, v in m.items(): f[k] = v
        tabs.append(f); infos.append(dict(**m, **info))
    except Exception as e: print('FAILED', r, repr(e))
    if i % 10 == 0: print(f'{i + 1}/{len(runs)} ({time.time() - t0:.0f}s)')
assert tabs, 'no run could be processed: read the FAILED lines above'
D = pd.concat(tabs, ignore_index=True); INFO = pd.DataFrame(infos)
print(INFO.groupby(['scenario', 'variant']).agg(runs=('run', 'count'), win=('n_win', 'sum'), attack_win=('attack_win', 'sum'), fault_win=('fault_win', 'sum'), labels_ok=('labels_match', 'all'), tz=('tz_hours', lambda x: sorted(set(x))), max_resid_s=('tz_resid_s', lambda x: np.abs(x).max())).to_string())
bad = INFO[(INFO.tz_resid_s.abs() > 5) | (~INFO.labels_match)]; print('\nruns needing attention (clock residual > 5 s or label/frame mismatch):', len(bad)); print(bad[['scenario', 'variant', 'run', 'tz_hours', 'tz_resid_s', 'labels_match']].to_string() if len(bad) else '')
print('\nwindow class counts:', D.y.value_counts().sort_index().to_dict(), '(0 normal, 1 fault, 2 attack)')


In [ ]:
ELEC = [c for c in D.columns if c.startswith('e_')]; HOST = [c for c in D.columns if c.startswith('h_')]; PKT = [c for c in D.columns if c.startswith('p_')]
GROUPS = {'all': ELEC + HOST + PKT, 'electrical': ELEC, 'host': HOST, 'packets': PKT, 'cyber (host+packets)': HOST + PKT, 'electrical+host': ELEC + HOST}
print({k: len(v) for k, v in GROUPS.items()})
tr_idx = np.flatnonzero(D.run.isin(TRAIN_RUNS).values); te_idx = np.flatnonzero(D.run.isin(TEST_RUNS).values)
print('held-out runs: train windows', len(tr_idx), '| test windows', len(te_idx), '| test class counts', np.bincount(D.y.values[te_idx], minlength=3))
def split_random(seed): return train_test_split(np.arange(len(D)), test_size=0.3, stratify=D.y.values, random_state=seed)
def fit_pred(name, Xtr, ytr, Xte, seed):
    if name == 'xgb': return xgb.XGBClassifier(**XGB_P, random_state=seed, num_class=3, objective='multi:softprob').fit(Xtr, ytr).predict(Xte)
    return RandomForestClassifier(300, n_jobs=-1, random_state=seed, class_weight='balanced').fit(Xtr, ytr).predict(Xte)
def score(y, p):
    atk = y == 2
    return dict(macro_f1=f1_score(y, p, average='macro', labels=[0, 1, 2]), f1_normal=f1_score(y, p, labels=[0], average='macro'), f1_fault=f1_score(y, p, labels=[1], average='macro') if (y == 1).any() else np.nan,
                f1_attack=f1_score(y, p, labels=[2], average='macro') if atk.any() else np.nan, attack_recall=float((p[atk] == 2).mean()) if atk.any() else np.nan, attack_false_alarm=float((p[~atk] == 2).mean()))
rows = []; PRED = {}
for s in SEEDS:
    for split in ['heldout_runs', 'random_windows']:
        a, b = (tr_idx, te_idx) if split == 'heldout_runs' else split_random(s)
        for model in ['xgb', 'rf']:
            for gname, cols in GROUPS.items():
                if model == 'rf' and gname not in ('all',): continue
                X = D[cols].values.astype('float32'); y = D.y.values; p = fit_pred(model, X[a], y[a], X[b], s)
                rows.append(dict(seed=s, split=split, model=model, features=gname, **score(y[b], p)))
                if model == 'xgb' and gname == 'all' and split == 'heldout_runs': PRED[s] = (b, p)
    print('seed', s, 'done')
R = pd.DataFrame(rows); ms = lambda x: f'{np.mean(x):.3f} +/- {np.std(x, ddof=1):.3f}' if len(x) > 1 else f'{np.mean(x):.3f}'
SUM = R.groupby(['split', 'model', 'features'])[['macro_f1', 'f1_attack', 'attack_recall', 'attack_false_alarm']].agg(ms); print(SUM.to_string())


In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
pr = []
for split in R.split.unique():
    piv = R[(R.split == split) & (R.model == 'xgb')].pivot(index='seed', columns='features', values='macro_f1'); fam = []
    for g_ in [k for k in GROUPS if k != 'all']:
        d_ = piv['all'] - piv[g_]; fam.append((g_, d_.mean(), stats.ttest_rel(piv['all'], piv[g_])[1] if len(SEEDS) > 1 and d_.std() > 0 else np.nan))
    adj = holm([np.nan_to_num(f[2], nan=1.0) for f in fam])
    for (g_, dm, p_), pa in zip(fam, adj): pr.append(dict(split=split, all_minus=g_, mean_diff=dm, p_raw=p_, p_holm=pa))
PAIR = pd.DataFrame(pr); print('\nall features minus group (macro-F1), paired over seeds, Holm within split:'); print(PAIR.round(4).to_string(index=False))
# 95% interval for held-out-run macro-F1 (xgb, all features): bootstrap over RUNS, averaged over seeds
tek = D.iloc[te_idx]; run_key = (tek.variant.astype(str) + '_' + tek.run.astype(str)).values; uk = np.unique(run_key); grp = {k: np.flatnonzero(run_key == k) for k in uk}
def boot(metric, n=1000, seed=0):
    rb = np.random.RandomState(seed); vals = []
    for _ in range(n):
        pick_ = rb.choice(uk, len(uk), replace=True); idx = np.concatenate([grp[k] for k in pick_]); yt = tek.y.values[idx]
        vals.append(np.nanmean([metric(yt, PRED[s][1][idx]) for s in SEEDS]))
    return np.nanpercentile(vals, [2.5, 97.5])
mf = lambda y, p: f1_score(y, p, average='macro', labels=[0, 1, 2]); ar = lambda y, p: float((p[y == 2] == 2).mean()) if (y == 2).any() else np.nan
BCI = dict(macro_f1=(float(np.mean([mf(tek.y.values, PRED[s][1]) for s in SEEDS])), *boot(mf)), attack_recall=(float(np.mean([ar(tek.y.values, PRED[s][1]) for s in SEEDS])), *boot(ar)))
print('\nheld-out runs, xgb all features, 95% bootstrap over runs:', {k: tuple(round(float(x), 3) for x in v) for k, v in BCI.items()})
perv = pd.DataFrame({'variant': tek.variant.values, 'y': tek.y.values, 'p': np.mean([PRED[s][1] for s in SEEDS], 0).round().astype(int)})
print('\nper-variant attack recall (held-out runs, seed-averaged predictions rounded):'); print(perv[perv.y == 2].groupby('variant').apply(lambda d: float((d.p == 2).mean())).round(3).to_string())


In [ ]:
T = {'models': R, 'summary': SUM.reset_index(), 'ablation_tests': PAIR, 'run_info': INFO}
for k_, t_ in T.items(): t_.to_csv(os.path.join(OUT, f'cpps_{k_}_{PIPELINE_VERSION}.csv'), index=False)
D.to_csv(os.path.join(OUT, f'cpps_windows_{PIPELINE_VERSION}.csv'), index=False)
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'tables': T, 'bootstrap_ci': BCI, 'groups': GROUPS, 'train_runs': sorted(TRAIN_RUNS), 'test_runs': sorted(TEST_RUNS), 'win_s': WIN}, os.path.join(OUT, f'cpps_results_{PIPELINE_VERSION}.pkl'))
print('saved to', OUT, sorted(os.listdir(OUT))[-8:])
